# 🔍 QARiB — Arabic Hate Speech Detection
### Before vs After Fine-Tuning Comparison

| Param | Value | Why |
|---|---|---|
| Model | QARiB | Pretrained on 420M Arabic tweets + MSA — built for social media Arabic |
| `MAX_LEN` | 80 | Covers 95%+ of Arabic social media posts; faster than 128 |
| `BATCH_SIZE` | 64/GPU | Safe in 16 GB T4 at seq=80; 2× throughput vs 32 |
| `GRAD_ACCUM` | 2 | Effective batch = 64×2×2GPUs = 256 |
| `EPOCHS` | 3 + early stop | Sufficient for BERT fine-tuning |
| `LR` | 3e-5 | Scaled for effective batch 256 |
| `LABEL_SMOOTH` | 0.1 | Regularisation, measurably improves F1 |
| LLRD decay | 0.9 | Per-layer LR — lower layers trained more conservatively |

**No pip installs — Kaggle pre-installed packages only.  
Works with any `transformers` version (4.x → 5.x).**

---
## 1. Imports & Environment

In [ ]:
import os, re, unicodedata, inspect, warnings, logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import transformers as _tf

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()

print(f'transformers : {_tf.__version__}')
print(f'torch        : {torch.__version__}')
print(f'device       : {device}  |  GPUs: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Version-safe TrainingArguments key (transformers >=4.41 renamed the param)
_ta_params         = set(inspect.signature(TrainingArguments.__init__).parameters)
EVAL_STRATEGY_KEY  = 'eval_strategy' if 'eval_strategy' in _ta_params else 'evaluation_strategy'
print(f'\neval key: "{EVAL_STRATEGY_KEY}"')

---
## 2. Configuration  ← set your paths here

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                    ★  USER CONFIGURATION  ★                     ║
# ╠══════════════════════════════════════════════════════════════════╣

# ── Dataset ─────────────────────────────────────────────────────────
# CSV with TEXT_COLUMN (Arabic text) + LABEL_COLUMN (0/1)
DATASET_PATH = '/kaggle/input/your-dataset/your_file.csv'   # ← SET PATH
TEXT_COLUMN  = 'text'                                        # ← column name
LABEL_COLUMN = 'label'                                       # ← 0=not hate, 1=hate

# ── Model ────────────────────────────────────────────────────────────
# QARiB: pretrained on 420M Arabic tweets + MSA (97 GB)
# Best published results on Arabic hate speech: 93-95% accuracy, 87-92% F1
MODEL_NAME   = 'qarib/bert-base-qarib'                       # ← or Kaggle model path

# ╠══════════════════════════════════════════════════════════════════╣
# ║               OPTIMISED HYPER-PARAMETERS                        ║
# ╠══════════════════════════════════════════════════════════════════╣
#
#  Param              Value   Reasoning
#  MAX_LEN              80    Arabic tweets = 20-60 tokens post-BPE; 80 covers
#                             95%+ while being 2.5× faster than MAX_LEN=128
#                             (attention is O(n²): 80²/128² = 0.39 cost)
#
#  BATCH_SIZE/GPU       64    BERT-base at seq=80 in fp16 uses ~4.5 GB VRAM;
#                             64 fits comfortably in 16 GB T4
#
#  GRAD_ACCUM            2    Effective batch = 64 × 2 × 2GPUs = 256;
#                             large batches → smoother gradients → better F1
#
#  EPOCHS                3    3 epochs is standard for BERT; early stopping
#                             (patience=1) cuts it short if val F1 plateaus
#
#  LEARNING_RATE       3e-5   Linear scaling: base 2e-5 × (256/128) ≈ 3e-5
#
#  WARMUP_RATIO        0.06   Fewer total steps → shorter warmup
#
#  LABEL_SMOOTH         0.1   Softens targets [0,1] → [0.05, 0.95];
#                             prevents overconfidence, improves macro-F1
#
#  LLRD decay           0.9   Layers closer to input get smaller LR;
#                             consistently adds +1-2% accuracy on fine-tuning
# ╚══════════════════════════════════════════════════════════════════╝

MAX_LEN       = 80
BATCH_SIZE    = 64      # per GPU
GRAD_ACCUM    = 2       # effective batch = 64 × 2 × 2 GPUs = 256
NUM_EPOCHS    = 3
LEARNING_RATE = 3e-5
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01
LABEL_SMOOTH  = 0.1
LLRD_DECAY    = 0.9
TEST_SIZE     = 0.15
VAL_SIZE      = 0.10

OUTPUT_DIR    = './qarib-finetuned'
LOGS_DIR      = './logs'

ID2LABEL = {0: 'Not Hate', 1: 'Hate Speech'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

eff = BATCH_SIZE * GRAD_ACCUM * max(n_gpus, 1)
print('Configuration ✅')
print(f'  MAX_LEN        : {MAX_LEN}')
print(f'  Effective batch: {eff}  ({BATCH_SIZE}/GPU × {GRAD_ACCUM} accum × {max(n_gpus,1)} GPUs)')
print(f'  EPOCHS         : {NUM_EPOCHS} + early stopping')
print(f'  LR / LLRD      : {LEARNING_RATE} / {LLRD_DECAY}')
print(f'  Label smoothing: {LABEL_SMOOTH}')

---
## 3. Arabic Normaliser & Tokenizer

In [ ]:
def normalize_arabic(text):
    """Arabic text normalisation for social media / tweet data.
    Mirrors the preprocessing used during QARiB pretraining:
    remove diacritics, normalise alef/ya/ta-marbuta variants.
    """
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)        # tashkeel
    text = re.sub(r'[إأآا]', 'ا', text)                              # alef variants
    text = re.sub(r'[ىي]',   'ي', text)                              # ya
    text = re.sub(r'ة',      'ه', text)                              # ta marbuta
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\s\d]', ' ', text)  # non-Arabic
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer ready  |  vocab: {tokenizer.vocab_size:,}')

# Sanity check
_t = 'أَنْتُمْ قَوْمٌ لا تَسْتَحِقُّونَ الْعَيْشَ بَيْنَنَا'
print(f'\nNormalization test:')
print(f'  raw  : {_t}')
print(f'  clean: {normalize_arabic(_t)}')
_enc = tokenizer(normalize_arabic(_t), return_tensors='pt')
print(f'  tokens: {_enc["input_ids"].shape[1]}')

---
## 4. Sample Texts (evaluated Before AND After fine-tuning)

In [ ]:
SAMPLE_TEXTS = [
    # Hate Speech
    'أنتم قوم لا تستحقون العيش بيننا',
    'هؤلاء المهاجرون يدمرون بلدنا ويسرقون وظائفنا',
    'يجب طرد كل الأجانب من هذه الأرض',
    'لا مكان للكفار في مجتمعنا الإسلامي',
    'هذه الفئة من الناس لا تساوي شيئا',
    # Not Hate
    'أحب الطبيعة والجبال والبحر',
    'يجب أن نتعاون معا لبناء مستقبل أفضل',
    'التنوع الثقافي يثري المجتمعات البشرية',
    'قرأت كتابا رائعا عن تاريخ الحضارة العربية',
    'التعليم هو المفتاح الحقيقي للتقدم والازدهار',
    # Ambiguous
    'هذا الشعب له عادات غريبة جدا',
    'لا أفهم ثقافتهم ولا أريد أن أفهمها',
]
SAMPLE_LABELS = [1, 1, 1, 1, 1,  0, 0, 0, 0, 0,  1, 1]
print(f'✅ {len(SAMPLE_TEXTS)} evaluation samples ready')

---
## 5. Shared Classes & Utilities

In [ ]:
# ── Dataset (BERT-based: includes token_type_ids) ───────────────────
class ArabicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = [normalize_arabic(t) for t in texts]
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids'      : enc['input_ids'].squeeze(),
            'attention_mask' : enc['attention_mask'].squeeze(),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long),
        }
        # QARiB is BERT-based → include token_type_ids
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze()
        return item


# ── Inference ───────────────────────────────────────────────────────
def predict_batch(model, texts, batch_size=32):
    model.eval()
    all_logits = []
    preprocessed = [normalize_arabic(t) for t in texts]
    for i in range(0, len(preprocessed), batch_size):
        enc = tokenizer(
            preprocessed[i:i+batch_size],
            padding=True, truncation=True,
            max_length=MAX_LEN, return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            m = model.module if hasattr(model, 'module') else model
            all_logits.append(m(**enc).logits.cpu())
    logits    = torch.cat(all_logits, dim=0)
    all_probs = torch.softmax(logits, dim=-1).numpy()
    return np.argmax(all_probs, 1).tolist(), np.max(all_probs, 1).tolist(), all_probs


# ── Pretty-print results ────────────────────────────────────────────
def show_results(texts, preds, confs, true_labels=None, title='Results'):
    rows = []
    for i, (t, p, c) in enumerate(zip(texts, preds, confs)):
        row = {'#': i+1, 'Text': (t[:52]+'…') if len(t)>52 else t,
               'Prediction': ID2LABEL[p], 'Confidence': f'{c:.1%}'}
        if true_labels is not None:
            row['Ground Truth'] = ID2LABEL[true_labels[i]]
            row['✓/✗'] = '✓' if p == true_labels[i] else '✗'
        rows.append(row)
    df = pd.DataFrame(rows).set_index('#')
    print(f'\n{"─"*8} {title} {"─"*8}')
    print(df.to_string())
    if true_labels is not None:
        print(f'\n  Accuracy: {accuracy_score(true_labels,preds):.1%}  '
              f'Macro-F1: {f1_score(true_labels,preds,average="macro",zero_division=0):.4f}')
    return df


# ── compute_metrics for Trainer ─────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy'    : float(accuracy_score(labels, preds)),
        'f1_macro'    : float(f1_score(labels, preds, average='macro',    zero_division=0)),
        'f1_weighted' : float(f1_score(labels, preds, average='weighted', zero_division=0)),
    }


# ── Layer-wise LR Decay (LLRD) ──────────────────────────────────────
def get_llrd_params(model, base_lr=LEARNING_RATE, decay=LLRD_DECAY):
    """Per-layer learning rate decay. Lower layers get smaller LR."""
    no_decay = ['bias', 'LayerNorm.weight']
    groups   = []
    # Classifier head → full LR
    groups.append({'params': [p for n,p in model.named_parameters() if 'classifier' in n],
                   'lr': base_lr, 'weight_decay': 0.0})
    # Encoder layers (top → bottom)
    n_layers = model.config.num_hidden_layers
    for i in range(n_layers-1, -1, -1):
        lr_i   = base_lr * (decay ** (n_layers - i))
        prefix = f'bert.encoder.layer.{i}.'
        groups.append({'params': [p for n,p in model.named_parameters()
                                   if prefix in n and not any(nd in n for nd in no_decay)],
                       'lr': lr_i, 'weight_decay': WEIGHT_DECAY})
        groups.append({'params': [p for n,p in model.named_parameters()
                                   if prefix in n and any(nd in n for nd in no_decay)],
                       'lr': lr_i, 'weight_decay': 0.0})
    # Embeddings → smallest LR
    emb_lr = base_lr * (decay ** (n_layers + 1))
    groups.append({'params': [p for n,p in model.named_parameters()
                               if 'embeddings' in n and not any(nd in n for nd in no_decay)],
                   'lr': emb_lr, 'weight_decay': WEIGHT_DECAY})
    groups.append({'params': [p for n,p in model.named_parameters()
                               if 'embeddings' in n and any(nd in n for nd in no_decay)],
                   'lr': emb_lr, 'weight_decay': 0.0})
    return groups


# ── Custom Trainer with label smoothing ────────────────────────────
class SmoothedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


# ── Fresh model loader ──────────────────────────────────────────────
def load_model():
    """Load QARiB with a fresh classification head.
    UNEXPECTED/MISSING keys in the load report are normal:
      UNEXPECTED = pretraining heads (MLM/NSP) — not needed for classification
      MISSING    = classifier.weight/bias — freshly init'd, trained by fine-tuning
    """
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    )


# ── TrainingArguments builder (version-safe) ────────────────────────
def make_args(outdir, epochs):
    return TrainingArguments(
        output_dir                  = outdir,
        logging_dir                 = LOGS_DIR,
        num_train_epochs            = epochs,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE * 2,
        gradient_accumulation_steps = GRAD_ACCUM,
        fp16                        = True,
        learning_rate               = LEARNING_RATE,
        warmup_ratio                = WARMUP_RATIO,
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = 'cosine',
        max_grad_norm               = 1.0,
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1_macro',
        greater_is_better           = True,
        save_total_limit            = 2,
        logging_steps               = 100,
        report_to                   = 'none',
        dataloader_num_workers      = 2,
        dataloader_pin_memory       = True,
        seed                        = SEED,
        **{EVAL_STRATEGY_KEY: 'epoch'},
    )

print('✅ All utilities ready')

---
## 6. PHASE 1 — QARiB *Before* Fine-Tuning
> LOAD REPORT warnings are **normal** — UNEXPECTED = pretraining heads discarded; MISSING = classifier head randomly initialised.

In [ ]:
print(f'Loading QARiB: {MODEL_NAME}')
model_before = load_model()
if n_gpus > 1:
    model_before = nn.DataParallel(model_before)
    print(f'DataParallel ({n_gpus} GPUs)')
model_before = model_before.to(device)
print(f'✅ {sum(p.numel() for p in model_before.parameters()):,} parameters on {device}')

In [ ]:
preds_before, confs_before, probs_before = predict_batch(model_before, SAMPLE_TEXTS)
df_before = show_results(SAMPLE_TEXTS, preds_before, confs_before,
                         true_labels=SAMPLE_LABELS, title='QARiB BEFORE Fine-Tuning')

---
## 7. Load & Preprocess Dataset

In [ ]:
print(f'Loading: {DATASET_PATH}')
df_raw = pd.read_csv(DATASET_PATH)
print(f'Raw: {df_raw.shape}  |  columns: {df_raw.columns.tolist()}')
print(df_raw[[TEXT_COLUMN, LABEL_COLUMN]].head(3))

df = df_raw[[TEXT_COLUMN, LABEL_COLUMN]].copy()
df.columns = ['text', 'label']
df = df.dropna().reset_index(drop=True)
df['label'] = df['label'].astype(int)
df['text']  = df['text'].apply(normalize_arabic)
df = df[df['text'].str.strip().ne('')].reset_index(drop=True)

print(f'\nClean: {df.shape}')
for lbl, cnt in df['label'].value_counts().rename(index=ID2LABEL).items():
    print(f'  {lbl:<12}: {cnt:,}  ({cnt/len(df):.1%})')

In [ ]:
df_tv, df_test = train_test_split(df, test_size=TEST_SIZE,
                                  random_state=SEED, stratify=df['label'])
df_train, df_val = train_test_split(df_tv,
                                    test_size=VAL_SIZE/(1-TEST_SIZE),
                                    random_state=SEED, stratify=df_tv['label'])

train_ds = ArabicDataset(df_train['text'].tolist(), df_train['label'].tolist(), tokenizer, MAX_LEN)
val_ds   = ArabicDataset(df_val['text'].tolist(),   df_val['label'].tolist(),   tokenizer, MAX_LEN)
test_ds  = ArabicDataset(df_test['text'].tolist(),  df_test['label'].tolist(),  tokenizer, MAX_LEN)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

---
## 8. Fine-Tuning

In [ ]:
print(f'Loading fresh QARiB for fine-tuning...')
model_ft  = load_model().to(device)
args      = make_args(OUTPUT_DIR, NUM_EPOCHS)
optimizer = AdamW(get_llrd_params(model_ft), lr=LEARNING_RATE, eps=1e-8, betas=(0.9, 0.999))

trainer = SmoothedTrainer(
    model           = model_ft,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    tokenizer       = tokenizer,
    compute_metrics = compute_metrics,
    optimizers      = (optimizer, None),
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)],
)

print(f'🚀 Fine-tuning  |  eff. batch={BATCH_SIZE*GRAD_ACCUM*max(n_gpus,1)}  '
      f'seq={MAX_LEN}  smoothing={LABEL_SMOOTH}')
trainer.train()

---
## 9. Test Set Evaluation

In [ ]:
test_out   = trainer.predict(test_ds)
test_preds = np.argmax(test_out.predictions, axis=-1)

print('📊 Test Set Metrics:')
for k, v in test_out.metrics.items():
    print(f'  {k:<35} {v:.4f}')

print('\n📋 Classification Report:')
print(classification_report(df_test['label'].tolist(), test_preds,
                            target_names=list(ID2LABEL.values())))

cm = confusion_matrix(df_test['label'].tolist(), test_preds)
print('Confusion Matrix:')
print(pd.DataFrame(cm,
    index  =[f'Actual: {ID2LABEL[i]}'    for i in range(2)],
    columns=[f'Predicted: {ID2LABEL[i]}' for i in range(2)]).to_string())

---
## 10. PHASE 2 — QARiB *After* Fine-Tuning

In [ ]:
model_after = trainer.model
if n_gpus > 1:
    model_after = nn.DataParallel(model_after)
model_after = model_after.to(device)

preds_after, confs_after, probs_after = predict_batch(model_after, SAMPLE_TEXTS)
df_after = show_results(SAMPLE_TEXTS, preds_after, confs_after,
                        true_labels=SAMPLE_LABELS, title='QARiB AFTER Fine-Tuning')

---
## 11. Before vs After Comparison

In [ ]:
acc_b = accuracy_score(SAMPLE_LABELS, preds_before)
acc_a = accuracy_score(SAMPLE_LABELS, preds_after)
f1_b  = f1_score(SAMPLE_LABELS, preds_before, average='macro', zero_division=0)
f1_a  = f1_score(SAMPLE_LABELS, preds_after,  average='macro', zero_division=0)

cmp = pd.DataFrame({
    'Text'        : [(t[:45]+'…') if len(t)>45 else t for t in SAMPLE_TEXTS],
    'Truth'       : [ID2LABEL[l] for l in SAMPLE_LABELS],
    'Before'      : [ID2LABEL[p] for p in preds_before],
    'Bef. Conf'   : [f'{c:.0%}'  for c in confs_before],
    'Bef. ✓/✗'    : ['✓' if p==g else '✗' for p,g in zip(preds_before, SAMPLE_LABELS)],
    'After'       : [ID2LABEL[p] for p in preds_after],
    'Aft. Conf'   : [f'{c:.0%}'  for c in confs_after],
    'Aft. ✓/✗'    : ['✓' if p==g else '✗' for p,g in zip(preds_after,  SAMPLE_LABELS)],
})
print('═'*125)
print('  QARiB — BEFORE vs AFTER Fine-Tuning')
print('═'*125)
print(cmp.to_string(index=False))
print(f'\n{"─"*52}')
print(f'{"Metric":<18} {"Before FT":>12} {"After FT":>12} {"Δ":>8}')
print(f'{"─"*52}')
print(f'{"Accuracy":<18} {acc_b:>12.1%} {acc_a:>12.1%} {acc_a-acc_b:>+8.1%}')
print(f'{"Macro-F1":<18} {f1_b:>12.4f} {f1_a:>12.4f} {f1_a-f1_b:>+8.4f}')
print(f'{"─"*52}')

In [ ]:
labels_s = [f'S{i+1}\n({"H" if SAMPLE_LABELS[i] else "N"})' for i in range(len(SAMPLE_TEXTS))]
x, w = np.arange(len(SAMPLE_TEXTS)), 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('QARiB Arabic Hate Speech — Before vs After Fine-Tuning',
             fontsize=14, fontweight='bold')

b1 = ax1.bar(x-w/2, confs_before, w, label='Before FT', color='#E07B54', alpha=0.85)
b2 = ax1.bar(x+w/2, confs_after,  w, label='After FT',  color='#1D9E75', alpha=0.85)
ax1.axhline(0.5, color='gray', linestyle='--', lw=1)
ax1.set_xticks(x); ax1.set_xticklabels(labels_s, fontsize=8)
ax1.set_ylim(0, 1.12); ax1.set_ylabel('Confidence')
ax1.set_title('Per-sample confidence  (H=Hate, N=Not Hate)'); ax1.legend()
for bar in list(b1)+list(b2):
    ax1.annotate(f'{bar.get_height():.0%}',
                 xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                 xytext=(0,3), textcoords='offset points',
                 ha='center', va='bottom', fontsize=7)

xi = np.array([0,1])
ax2.bar(xi-w/2, [acc_b, acc_a], w, label='Accuracy', color='#E07B54', alpha=0.85)
ax2.bar(xi+w/2, [f1_b,  f1_a],  w, label='Macro-F1', color='#1D9E75', alpha=0.85)
ax2.set_xticks(xi); ax2.set_xticklabels(['Before FT', 'After FT'])
ax2.set_ylim(0, 1.1); ax2.set_ylabel('Score')
ax2.set_title('Overall Accuracy & F1'); ax2.legend()

plt.tight_layout()
plt.savefig('qarib_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → qarib_results.png')

---
## 12. Save Model

In [ ]:
SAVE_PATH = './qarib-hate-speech-arabic'
m = trainer.model
if hasattr(m, 'module'): m = m.module
m.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ Model saved → {SAVE_PATH}')